# T7.7 — 100-Species Fine-Grained Animal Classifier

**Dataset:** `ViratGarg/animal_species_SMAI` (HuggingFace) — 26.7k images · 100 species

| Step | What happens |
|------|--------------|
| 1 | Install dependencies |
| 2 | Imports & config |
| 3 | Download dataset from HuggingFace + stratified split |
| 4 | PyTorch Dataset wrapper + transforms |
| 5 | Build ResNet-50 model |
| 6 | Metrics (accuracy + macro-specificity) |
| 7 | Training loop |
| 8 | Run training & save outputs |

## Step 1 — Install Dependencies
Run once. Skip if already installed.

In [1]:
!pip install torch torchvision datasets scikit-learn Pillow tqdm --quiet

## Step 2 — Imports & Config
Edit config values here if needed (batch size, epochs, backbone, etc.)

In [2]:
import os, json, csv, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── CONFIG — edit here only ──────────────────────────────────
HF_DATASET  = "ViratGarg/animal_species_SMAI"
OUTPUT_DIR = "/content/drive/MyDrive/smai_outputs"
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 1e-4
IMG_SIZE    = 224
BACKBONE    = "resnet50"   # or "efficientnet_b0"
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Device: cuda


In [4]:
# ── Reproducibility ──────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Step 3 — Download Dataset & Stratified Split
Downloads from HuggingFace (cached after first run) and splits 80% train / 10% val / 10% test.
Every species appears in every split.

In [5]:
!git clone https://huggingface.co/datasets/ViratGarg/animal_species_SMAI


Cloning into 'animal_species_SMAI'...
remote: Enumerating objects: 26611, done.
remote: Total 26611 (delta 0), reused 0 (delta 0), pack-reused 26611 (from 1)
Receiving objects: 100% (26611/26611), 3.71 MiB | 8.53 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (26661/26661), done.
Filtering content: 100% (26659/26659), 1.86 GiB | 9.13 MiB/s, done.


In [6]:
def load_and_split(hf_dataset, val_ratio, test_ratio, seed):
    print("Loading dataset from local clone...")
    raw = load_dataset(
    "imagefolder",
    data_dir="./animal_species_SMAI",
    split="train"
)
    print(raw.features)


    if hasattr(raw.features["label"], "names"):
        species_names = raw.features["label"].names
    else:
        unique_ids = sorted(set(raw["label"]))
        species_names = [str(i) for i in unique_ids]

    label_map   = {name: idx for idx, name in enumerate(species_names)}
    num_classes = len(species_names)
    print(f"  Total samples : {len(raw)}")
    print(f"  Species found : {num_classes}")

    indices = list(range(len(raw)))
    labels  = raw["label"]

    train_idx, temp_idx = train_test_split(
        indices, test_size=(val_ratio + test_ratio),
        stratify=labels, random_state=seed
    )
    val_size_relative = val_ratio / (val_ratio + test_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=(1 - val_size_relative),
        stratify=[labels[i] for i in temp_idx], random_state=seed
    )

    train_ds = raw.select(train_idx)
    val_ds   = raw.select(val_idx)
    test_ds  = raw.select(test_idx)

    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_ds, val_ds, test_ds, label_map, species_names


train_hf, val_hf, test_hf, label_map, species_names = load_and_split(
    HF_DATASET, VAL_RATIO, TEST_RATIO, SEED
)
num_classes = len(label_map)

# Save label map immediately (teammate needs this)
label_map_path = os.path.join(OUTPUT_DIR, "label_map.json")
with open(label_map_path, "w") as f:
    json.dump(label_map, f, indent=2)
print(f"\nlabel_map.json saved → {label_map_path}")

Loading dataset from local clone...


Resolving data files:   0%|          | 0/26659 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['ADONIS', 'AFRICAN GIANT SWALLOWTAIL', 'AMERICAN SNOOT', 'AN 88', 'APPOLLO', 'ARCIGERA FLOWER MOTH', 'ATALA', 'ATLAS MOTH', 'Asian Green Bee-Eater', 'BANDED ORANGE HELICONIAN', 'BANDED PEACOCK', 'BANDED TIGER MOTH', 'BECKERS WHITE', 'BIRD CHERRY ERMINE MOTH', 'BLACK HAIRSTREAK', 'BLUE MORPHO', 'BLUE SPOTTED CROW', 'BROOKES BIRDWING', 'BROWN ARGUS', 'BROWN SIPROETA', 'Brown-Headed Barbet', 'CABBAGE WHITE', 'CAIRNS BIRDWING', 'CHALK HILL BLUE', 'CHECQUERED SKIPPER', 'CHESTNUT', 'CINNABAR MOTH', 'CLEARWING MOTH', 'CLEOPATRA', 'CLODIUS PARNASSIAN', 'CLOUDED SULPHUR', 'COMET MOTH', 'Cattle Egret', 'Common Kingfisher', 'Common Myna', 'Common Rosefinch', 'Common Tailorbird', 'Coppersmith Barbet', 'Forest Wagtail', 'Gray Wagtail', 'Hoopoe', 'House Crow', 'Indian Grey Hornbill', 'Indian Peacock', 'Indian Pitta', 'Indian Roller', 'Jungle Babbler', 'Northern Lapwing', 'Red-Wattled Lapwing', 'Ruddy Shelduck', 'Rufous Treepie', 'Sa

## Step 4 — PyTorch Dataset Wrapper & Transforms

In [7]:
class HFSpeciesDataset(Dataset):
    """Wraps a HuggingFace dataset shard. Each row: {image: PIL.Image, label: int}"""
    def __init__(self, hf_shard, transform=None):
        self.data      = hf_shard
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data[idx]
        image = row["image"]
        label = int(row["label"])
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [8]:
def get_transforms(img_size):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    return {
        "train": transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]),
        "eval": transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]),
    }

tfm = get_transforms(IMG_SIZE)

train_loader = DataLoader(
    HFSpeciesDataset(train_hf, tfm["train"]),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    HFSpeciesDataset(val_hf, tfm["eval"]),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    HFSpeciesDataset(test_hf, tfm["eval"]),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"DataLoaders ready — batch size: {BATCH_SIZE}")

DataLoaders ready — batch size: 32


## Step 5 — Build Model
Pretrained ResNet-50 with the final layer replaced by a 100-class head.

In [9]:
def build_model(backbone, num_classes):
    if backbone == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(model.fc.in_features, num_classes)
        )
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_feat = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feat, num_classes)
        )
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return model


model     = build_model(BACKBONE, num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Model: {BACKBONE} | Output classes: {num_classes} | Device: {DEVICE}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


Model: resnet50 | Output classes: 100 | Device: cuda


## Step 6 — Metrics
Accuracy and macro-specificity (matches teammate's notebook format).

In [10]:
def calculate_metrics(y_true, y_pred, num_classes):
    acc = accuracy_score(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specificities = []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp
        denom = tn + fp
        specificities.append(tn / denom if denom > 0 else 0.0)
    return float(acc), float(np.mean(specificities))


def run_epoch(model, loader, criterion, optimizer, device, num_classes, phase):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.set_grad_enabled(is_train):
        for images, labels in tqdm(loader, desc=f"  {phase:8s}", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss  = total_loss / len(loader.dataset)
    acc, spec = calculate_metrics(y_true, y_pred, num_classes)
    return avg_loss, acc, spec

print("Metric functions defined.")

Metric functions defined.


## Step 7 — Training Loop
Trains for `NUM_EPOCHS` epochs. Saves the best model and a checkpoint every epoch.

In [11]:
best_val_acc = 0.0
metrics_rows = []

print(f"{'='*50}")
print(f"  T7.7 — 100-Species Classifier")
print(f"  Device : {DEVICE}")
print(f"{'='*50}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  (lr={scheduler.get_last_lr()[0]:.2e})")

    tr_loss, tr_acc, tr_spec = run_epoch(model, train_loader, criterion,
                                         optimizer, DEVICE, num_classes, "train")
    vl_loss, vl_acc, vl_spec = run_epoch(model, val_loader,   criterion,
                                         None,      DEVICE, num_classes, "val")
    te_loss, te_acc, te_spec = run_epoch(model, test_loader,  criterion,
                                         None,      DEVICE, num_classes, "test")
    scheduler.step()

    print(f"  Train → loss:{tr_loss:.4f}  acc:{tr_acc:.4f}  spec:{tr_spec:.4f}")
    print(f"  Val   → loss:{vl_loss:.4f}  acc:{vl_acc:.4f}  spec:{vl_spec:.4f}")
    print(f"  Test  → loss:{te_loss:.4f}  acc:{te_acc:.4f}  spec:{te_spec:.4f}")

    metrics_rows.append({
        "epoch":      epoch,
        "train_loss": round(tr_loss, 4), "train_acc": round(tr_acc, 4), "train_spec": round(tr_spec, 4),
        "val_loss":   round(vl_loss, 4), "val_acc":   round(vl_acc, 4), "val_spec":   round(vl_spec, 4),
        "test_loss":  round(te_loss, 4), "test_acc":  round(te_acc, 4), "test_spec":  round(te_spec, 4),
    })

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"  ✓ Best model saved (val_acc={vl_acc:.4f})")

    torch.save(model.state_dict(),
               os.path.join(OUTPUT_DIR, f"checkpoint_epoch_{epoch:02d}.pth"))
    print()

  T7.7 — 100-Species Classifier
  Device : cuda

Epoch 01/20  (lr=1.00e-04)


  Train → loss:1.7064  acc:0.7809  spec:0.9978
  Val   → loss:0.9931  acc:0.9437  spec:0.9994
  Test  → loss:1.0117  acc:0.9411  spec:0.9994
  ✓ Best model saved (val_acc=0.9437)

Epoch 02/20  (lr=9.94e-05)


  Train → loss:1.0079  acc:0.9461  spec:0.9995
  Val   → loss:0.9438  acc:0.9587  spec:0.9996
  Test  → loss:0.9561  acc:0.9546  spec:0.9995
  ✓ Best model saved (val_acc=0.9587)

Epoch 03/20  (lr=9.76e-05)


  Train → loss:0.9366  acc:0.9655  spec:0.9997
  Val   → loss:0.9166  acc:0.9644  spec:0.9996
  Test  → loss:0.9406  acc:0.9572  spec:0.9996
  ✓ Best model saved (val_acc=0.9644)

Epoch 04/20  (lr=9.46e-05)


  Train → loss:0.8969  acc:0.9762  spec:0.9998
  Val   → loss:0.9318  acc:0.9565  spec:0.9996
  Test  → loss:0.9407  acc:0.9587  spec:0.9996

Epoch 05/20  (lr=9.05e-05)


...

  Train → loss:0.7992  acc:0.9992  spec:1.0000
  Val   → loss:0.8828  acc:0.9737  spec:0.9997
  Test  → loss:0.8896  acc:0.9696  spec:0.9997
  ✓ Best model saved (val_acc=0.9737)

Epoch 17/20  (lr=9.55e-06)


  Train → loss:0.7980  acc:0.9990  spec:1.0000
  Val   → loss:0.8816  acc:0.9730  spec:0.9997
  Test  → loss:0.8892  acc:0.9715  spec:0.9997

Epoch 18/20  (lr=5.45e-06)


  Train → loss:0.7971  acc:0.9987  spec:1.0000
  Val   → loss:0.8762  acc:0.9752  spec:0.9997
  Test  → loss:0.8884  acc:0.9704  spec:0.9997
  ✓ Best model saved (val_acc=0.9752)

Epoch 19/20  (lr=2.45e-06)


  Train → loss:0.7962  acc:0.9991  spec:1.0000
  Val   → loss:0.8795  acc:0.9741  spec:0.9997
  Test  → loss:0.8874  acc:0.9719  spec:0.9997

Epoch 20/20  (lr=6.16e-07)


  Train → loss:0.7955  acc:0.9996  spec:1.0000
  Val   → loss:0.8772  acc:0.9741  spec:0.9997
  Test  → loss:0.8863  acc:0.9722  spec:0.9997



## Step 8 — Save Metrics CSV

In [12]:
csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=metrics_rows[0].keys())
    writer.writeheader()
    writer.writerows(metrics_rows)

print(f"metrics.csv saved → {csv_path}")
print(f"Best val accuracy : {best_val_acc:.4f}")
print(f"\nAll outputs in: {os.path.abspath(OUTPUT_DIR)}/")
print("  best_model.pth")
print("  label_map.json   ← share this with the combining teammate")
print("  metrics.csv")
print("  checkpoint_epoch_XX.pth  (one per epoch)")

metrics.csv saved → /content/drive/MyDrive/smai_outputs/metrics.csv
Best val accuracy : 0.9752

All outputs in: /content/drive/MyDrive/smai_outputs/
  best_model.pth
  label_map.json   ← share this with the combining teammate
  metrics.csv
  checkpoint_epoch_XX.pth  (one per epoch)
